# Run all configurations and params to find the best models

Runs various combinations of data sources and hyperparameters to find the best-performing conflict escalation model, logging each run to MLflow.

For every combination of included data sources (food prices, rainfall, ACLED text embeddings), escalation threshold `k`, and event column (`event_type` or `sub_event_type`), the notebook:

1. Loads and combines the corresponding cleaned dataset via `get_clean_combined_data`.
2. Trains and evaluates an XGBoost classifier for each number of cross-validation splits (`n`), using `train_evaluate_model` with a randomised hyperparameter search over `xgb_params`.
3. Logs the resulting metrics, parameters, and tags to MLflow, and appends a backup row to a local CSV (`evaluation/{COUNTRY}_results.csv`) in case MLflow logging fails.
4. Records each completed run in `models/completed_runs.txt` so that re-running the notebook skips runs that have already finished, making the sweep resumable.

**WARNING: This file takes a long time to run and runs hundreds of models. Use [run_best_model](models/run_best_models.py) to access run only the best model config and params**

In [1]:
import itertools
import logging
import os
from pathlib import Path

import mlflow
import pandas as pd
from dotenv import load_dotenv

from models.train_models import train_evaluate_model
from utils.constants import COUNTRY
from utils.data_prep import get_clean_combined_data

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

load_dotenv()

tracking_uri = os.environ["MLFLOW_TRACKING_URI"]
mlflow.set_tracking_uri(tracking_uri)
logger.info(
    f"MLflow tracking URI set to: {mlflow.get_tracking_uri()}"
)  # mlflow server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000

mlflow.sklearn.autolog(disable=True)

INFO:__main__:MLflow tracking URI set to: http://127.0.0.1:5000


In [2]:
completed_runs_file = "models/completed_runs.txt"
local_backup_file = Path(f"evaluation/{COUNTRY.lower()}_results_redo.csv")

if os.path.exists(completed_runs_file):
    with open(completed_runs_file, "r") as f:
        completed_runs = {line.strip() for line in f if line.strip()}
    with open(completed_runs_file, "w") as f:
        f.write("\n".join(sorted(completed_runs)) + "\n")
else:
    completed_runs = set()

print(f"Loaded {len(completed_runs)} completed runs from memory.")

Loaded 0 completed runs from memory.


In [3]:
xgb_params = {
    "max_depth": [3, 5, 7],
    "min_child_weight": [1, 3, 5],
    "max_delta_step": [0, 1, 5],
    "gamma": [0, 1, 3, 5],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_alpha": [0, 0.1, 1, 2],
    "reg_lambda": [1, 5, 10],
    "colsample_bylevel": [0.6, 0.8, 1.0],
}

In [4]:
# ks = [0.25, 0.5, 0.75, 1, 1.25, 1.5, 1.6, 1.65, 1.75, 2, 2.5]
# ns = [4, 5]
# event_cols = ["sub_event_type", "event_type"]
# include_food_options = [True, False]
# include_rain_options = [True, False]
# include_text_options = [True, False]
# conflict_only_embedding_options = [True, False]
# food_recency_options = [True, False]
# search_seeds = [23] # 32, 111, 2025, 999

# THRESHOLD_FIX_APPLIED = True

In [5]:
ks = [0.25, 0.5, 1]
ns = [4, 5]
event_cols = ["sub_event_type", "event_type"]
include_food_options = [True, False]
include_rain_options = [True, False]
include_text_options = [True, False]
conflict_only_embedding_options = [True, False]
food_recency_options = [False]
search_seeds = [23] # 32, 111, 2025, 999

THRESHOLD_FIX_APPLIED = False

In [6]:
data_configs = itertools.product(
    include_food_options,
    include_rain_options,
    include_text_options,
    ks,
    event_cols,
    search_seeds
)

for (
    include_food,
    include_rain,
    include_text,
    k,
    event_col,
    seed
) in data_configs:
    food_str = "_food" if include_food else ""
    rain_str = "_rain" if include_rain else ""
    event_str = "event" if event_col == "event_type" else "sub"
    thresh_str = "_threshold_change" if THRESHOLD_FIX_APPLIED else ""

    if include_text:
        pca_options = [
            True,
            False,
        ]  # Non-PCA embeddings took a lot longer but keeping this in here to show that both were run
        conflict_only_options = conflict_only_embedding_options
    else:
        pca_options = [False]  # Only run without PCA when text isn't included
        conflict_only_options = [None]  # Not applicable when text isn't included

    if include_food:
        price_recency_options = (
            food_recency_options  # Flag for whether to include staleness column
        )
    else:
        price_recency_options = [None]

    for conflict_only in conflict_only_options:
        if include_text:
            text_str = "_text_conflict" if conflict_only else "_text_all"
        else:
            text_str = ""

        for price in price_recency_options:
            price_string = "_price-recency" if price else ""

            which_data = f"acled_{event_str}{food_str}{rain_str}{text_str}{price_string}{thresh_str}_{seed}"

            all_runs_completed = True
            for n in ns:
                for use_pca in pca_options:
                    pca_str = "_pca" if use_pca else ""
                    expected_run = f"{which_data}{pca_str}_{k}_{n}"
                    if expected_run not in completed_runs:
                        all_runs_completed = False
                        break  # Stop checking this inner loop if we find a missing run
                if not all_runs_completed:
                    break  # Stop checking the outer loop too

            if all_runs_completed:
                print(
                    f"Skipping data load for {which_data} - all associated runs are complete."
                )
                continue

            # Only load data if there is at least one missing run
            data_sources = [
                src
                for src, include in zip(
                    ["food", "rain", "text"], [include_food, include_rain, include_text]
                )
                if include
            ]

            model_data, predictor_cols = get_clean_combined_data(
                data_sources=data_sources,
                k=k,
                event_col=event_col,
                conflict_only_embeddings=bool(conflict_only),
                price_recency=bool(price),
            )

            for n in ns:
                for use_pca in pca_options:
                    pca_str = "_pca" if use_pca else ""
                    run_name = f"{which_data}{pca_str}_{k}_{n}"

                    if run_name in completed_runs:
                        print(f"Skipping already completed run: {run_name}")
                        continue

                    all_params = {
                        **xgb_params,
                        "k": k,
                        "event_col": event_col,
                        "n_splits": n,
                        "use_pca": use_pca,
                        "seed": seed
                    }

                    with mlflow.start_run(run_name=run_name) as active_run:
                        mlflow.set_tags(
                            {
                                "data_version": which_data,
                                "remove_abyei": True,
                                "threshold_fix_applied": THRESHOLD_FIX_APPLIED,
                                "include_food": include_food,
                                "include_rain": include_rain,
                                "include_text": include_text,
                                "conflict_only_embeddings": bool(conflict_only),
                                "price_recency": bool(price),
                                "use_pca": use_pca,
                                "k": k,
                                "n_splits": n,
                                "event_col": event_col,
                                "seed": seed
                            }
                        )
                        logger.info(f"Running mode: {run_name}")

                        results, best_params, _, onset_preds = train_evaluate_model(
                            model_data,
                            predictor_cols,
                            all_params,
                            best_params=False,
                            use_pca=use_pca,
                            compute_shap=False,  # Only compute shap on best params,
                            threshold_fix=THRESHOLD_FIX_APPLIED,
                            return_onset_predictions=True
                        )
                        onset_preds.to_csv(
                            f"evaluation/model_reports/{run_name}_onset.csv",
                            index=False,
                        )
                        # Back up data locally as well as to mlruns
                        backup_row = {
                            "run_name": run_name,
                            "run_id": active_run.info.run_id,
                            "data_version": which_data,
                            "threshold_fix_applied": THRESHOLD_FIX_APPLIED,
                            "include_food": include_food,
                            "include_rain": include_rain,
                            "include_text": include_text,
                            "conflict_only_embeddings": bool(conflict_only),
                            "use_pca": use_pca,
                            "k": k,
                            "n_splits": n,
                            "event_col": event_col,
                            "price_recency": bool(price),
                            **results,
                            **{f"param_{pk}": pv for pk, pv in best_params.items()},
                            "seed": seed
                        }
                        backup_df = pd.DataFrame([backup_row])
                        write_header = not local_backup_file.exists()

                        if not write_header and local_backup_file.stat().st_size > 0:
                            with open(local_backup_file, "rb") as f:
                                f.seek(-1, os.SEEK_END)
                                if f.read(1) != b"\n":
                                    with open(local_backup_file, "a") as f2:
                                        f2.write("\n")

                        backup_df.to_csv(
                            local_backup_file,
                            mode="a",
                            header=write_header,
                            index=False,
                        )

                        try:
                            mlflow.log_params(best_params)
                            mlflow.log_metrics(
                                {key: float(val) for key, val in results.items()}
                            )
                            mlflow.log_dict(results, "model_report.json")

                            verify_run = mlflow.get_run(active_run.info.run_id)
                            if not verify_run.data.metrics:
                                raise RuntimeError(
                                    f"mlflow logged no error but metrics are empty on "
                                    f"readback for run {run_name} - tracking store may "
                                    f"be silently failing again."
                                )
                        except Exception as e:
                            logger.error(
                                f"MLflow logging failed or did not verify for "
                                f"{run_name}: {e}. Results are still safe in "
                                f"{local_backup_file}."
                            )

                        completed_runs.add(run_name)  # Add to log file
                        with open(completed_runs_file, "a") as f:
                            f.write(run_name + "\n")

INFO:AcledClient:Access token correctly retrieved.
INFO:AcledClient:Reading data from data/acled/acled_sudan.csv.
INFO:Events processing:Data grouped by sub_event_type
INFO:Events processing:Escalation target set at 0.25 standard deviations above the mean.
INFO:Data preparation:ACLED data processed.
INFO:hdx.api.configuration:No HDX base configuration parameter. Using default base configuration file: /Users/evie.jones/Downloads/msc-final-project/.venv/lib/python3.14/site-packages/hdx/api/hdx_base_configuration.yaml.
INFO:hdx.api.configuration:Loading HDX base configuration from: /Users/evie.jones/Downloads/msc-final-project/.venv/lib/python3.14/site-packages/hdx/api/hdx_base_configuration.yaml
INFO:hdx.api.configuration:No HDX configuration parameter and no configuration file at default path: /Users/evie.jones/.hdx_configuration.yaml.
INFO:hdx.api.configuration:Read only access to HDX: True
INFO:HDXClient:Reading local file: data/hdx/hdx_sudan_food_prices.csv
INFO:Name mapping:Renamed 

--- Fold 1 ---
Train window: 2018-01 to 2018-12 (216 rows)
Test window:  2019-01 to 2019-12 (216 rows)
------------------------------
--- Fold 2 ---
Train window: 2018-01 to 2019-12 (432 rows)
Test window:  2020-01 to 2020-12 (216 rows)
------------------------------
--- Fold 3 ---
Train window: 2018-01 to 2020-12 (648 rows)
Test window:  2021-01 to 2021-12 (216 rows)
------------------------------
--- Fold 4 ---
Train window: 2018-01 to 2021-12 (864 rows)
Test window:  2022-01 to 2022-12 (216 rows)
------------------------------


INFO:__main__:Running mode: acled_sub_food_rain_text_conflict_23_0.25_4
INFO:Cross validation:Cross-validation testing splits:


🏃 View run acled_sub_food_rain_text_conflict_23_pca_0.25_4 at: http://127.0.0.1:5000/#/experiments/0/runs/dbfab7858f6b4666bdcc3a378566457a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0
--- Fold 1 ---
Train window: 2018-01 to 2018-12 (216 rows)
Test window:  2019-01 to 2019-12 (216 rows)
------------------------------
--- Fold 2 ---
Train window: 2018-01 to 2019-12 (432 rows)
Test window:  2020-01 to 2020-12 (216 rows)
------------------------------
--- Fold 3 ---
Train window: 2018-01 to 2020-12 (648 rows)
Test window:  2021-01 to 2021-12 (216 rows)
------------------------------
--- Fold 4 ---
Train window: 2018-01 to 2021-12 (864 rows)
Test window:  2022-01 to 2022-12 (216 rows)
------------------------------


/Users/evie.jones/Downloads/msc-final-project/.venv/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
INFO:__main__:Running mode: acled_sub_food_rain_text_conflict_23_pca_0.25_5
INFO:Text processing:PCA fit on train embeddings only: 768 raw dims -> 24 components for 90% variance.
INFO:Cross validation:Cross-validation testing splits:


...

INFO:__main__:Running mode: acled_sub_food_text_conflict_23_1_5
INFO:Cross validation:Cross-validation testing splits:


🏃 View run acled_sub_food_text_conflict_23_pca_1_5 at: http://127.0.0.1:5000/#/experiments/0/runs/eaae576e99ca4e3d91ec92c3f9c14349
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0
--- Fold 1 ---
Train window: 2018-01 to 2018-10 (180 rows)
Test window:  2018-11 to 2019-08 (180 rows)
------------------------------
--- Fold 2 ---
Train window: 2018-01 to 2019-08 (360 rows)
Test window:  2019-09 to 2020-06 (180 rows)
------------------------------
--- Fold 3 ---
Train window: 2018-01 to 2020-06 (540 rows)
Test window:  2020-07 to 2021-04 (180 rows)
------------------------------
--- Fold 4 ---
Train window: 2018-01 to 2021-04 (720 rows)
Test window:  2021-05 to 2022-02 (180 rows)
------------------------------
--- Fold 5 ---
Train window: 2018-01 to 2022-02 (900 rows)
Test window:  2022-03 to 2022-12 (180 rows)
------------------------------


/Users/evie.jones/Downloads/msc-final-project/.venv/lib/python3.14/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
Exception ignored while calling ctypes callback function <bound method DataIter._next_wrapper of <xgboost.core.SingleBatchInternalIter object at 0x125f76490>>:
Traceback (most recent call last):
  File "/Users/evie.jones/Downloads/msc-final-project/.venv/lib/python3.14/site-packages/xgboost/core.py", line 425, in _next_wrapper
    def _next_wrapper(self, this: None) -> int:  # pylint: disable=unused-argument
KeyboardInterrupt: 
Process LokyProcess-3764:
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.14/3.14.6/Frameworks/Python.framework/Versions/3.14/lib/python3.14/multiprocessing/process.py", line 320, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/opt/homebrew/Cellar/p

🏃 View run acled_sub_food_text_conflict_23_1_5 at: http://127.0.0.1:5000/#/experiments/0/runs/3487c8e0d3d5482eaee9209809c84bb2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/0


KeyboardInterrupt: 